In [ ]:
# torch imports --

import torch
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary

# others --

from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
from src.tokenizer import CharTokenizer
from src.models import Mlm_RM

In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

#### Building the data

In [ ]:
# load text -- 
with open("data/shakespeare_preprocessed.txt", "r", encoding="utf-8") as f:
    text = f.read()

# defining the tokenizer --
tokenizer = CharTokenizer(text)

In [ ]:
context_window = 64
batch_size = 64

X, y = tokenizer.encode(text, context_window=context_window, sequential=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#### Building the model

In [ ]:
# defining hyperparameters
embedding_dim = 32
hidden_size = 256
num_layers = 2

num_epochs = 1000
learning_rate = 0.001

In [ ]:
model = Mlm_RM(
    model_type='lstm',
    vocab_size= len(tokenizer.vocabulary),
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    device=device
)

model.load_state_dict(torch.load('parameters/rm_model.pth', map_location=device))

model.eval()


In [ ]:
model.fit(
    train_loader=train_loader,
    test_loader=test_loader,
)